# Inspect clinic.db

In [6]:
%load_ext autoreload
%autoreload 2

import sqlite3
from pathlib import Path

import pandas as pd

# notebook lives in root/notebooks; the db lives in root/data
DB_PATH = Path("../data/clinic.db")

assert DB_PATH.exists(), f"Could not find {DB_PATH.resolve()} - update DB_PATH above"

conn = sqlite3.connect(DB_PATH)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## Tables in the database

In [7]:
tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn
)["name"].tolist()
tables

['agent_jobs', 'calendar', 'messages', 'patients', 'waitlist']

## Schema + row count per table

In [8]:
for table in tables:
    schema = pd.read_sql(f"PRAGMA table_info({table})", conn)
    row_count = pd.read_sql(f"SELECT COUNT(*) AS n FROM {table}", conn)["n"].iloc[0]
    print(f"--- {table} ({row_count} rows) ---")
    display(schema[["name", "type"]])
    print()

--- agent_jobs (0 rows) ---


,name,type
0,job_id,INTEGER
1,job_type,VARCHAR
2,appointment_id,INTEGER
3,status,VARCHAR(11)
4,matched_waitlist_id,INTEGER
5,matched_patient_id,INTEGER
6,created_at,DATETIME
7,updated_at,DATETIME



--- calendar (48 rows) ---


,name,type
0,appointment_id,INTEGER
1,patient_id,INTEGER
2,appointment_date,DATE
3,start_time,VARCHAR
4,end_time,VARCHAR
5,appointment_type,VARCHAR
6,status,VARCHAR(9)
7,created_at,DATETIME
8,canceled_at,DATETIME



--- messages (0 rows) ---


,name,type
0,message_id,INTEGER
1,patient_id,INTEGER
2,sender,VARCHAR(7)
3,message_text,VARCHAR
4,sent_at,DATETIME



--- patients (98 rows) ---


,name,type
0,patient_id,INTEGER
1,first_name,VARCHAR
2,last_name,VARCHAR
3,date_of_birth,DATE
4,phone,VARCHAR
5,email,VARCHAR
6,created_at,DATETIME



--- waitlist (40 rows) ---


,name,type
0,waitlist_id,INTEGER
1,patient_id,INTEGER
2,appointment_type,VARCHAR
3,status,VARCHAR(7)
4,added_at,DATETIME


## Preview each table

In [9]:
dfs = {table: pd.read_sql(f"SELECT * FROM {table}", conn) for table in tables}

for table, df in dfs.items():
    print(f"--- {table} ---")
    display(df.head(10))
    print()

--- agent_jobs ---


,job_id,job_type,appointment_id,status,matched_waitlist_id,matched_patient_id,created_at,updated_at



--- calendar ---


,appointment_id,patient_id,appointment_date,start_time,end_time,appointment_type,status,created_at,canceled_at
0,1,86,2026-09-01,09:00,09:30,Follow-up,canceled,2026-08-07 01:36:29.114824,2026-08-27 12:45:49.660615
1,2,60,2026-09-01,09:30,10:00,Physical Exam,completed,2026-08-06 03:08:28.457067,None
2,3,63,2026-09-01,10:00,10:30,Skin Screening,scheduled,2026-08-10 11:43:59.536022,None
3,4,10,2026-09-01,10:30,11:00,Consultation,scheduled,2026-08-18 08:52:09.190583,None
4,5,34,2026-09-01,11:00,11:30,Follow-up,scheduled,2026-08-02 09:21:38.140888,None
5,6,27,2026-09-01,11:30,12:00,Skin Screening,scheduled,2026-08-10 10:20:09.839687,None
6,7,2,2026-09-01,12:00,12:30,Vaccination,scheduled,2026-08-10 12:34:41.075270,None
7,8,69,2026-09-01,12:30,13:00,Vaccination,canceled,2026-08-18 15:14:19.149894,2026-08-27 11:46:34.284055
8,9,16,2026-09-01,13:00,13:30,Skin Screening,scheduled,2026-08-23 23:51:16.077608,None
9,10,61,2026-09-01,13:30,14:00,Vaccination,completed,2026-08-15 09:18:44.873797,None



--- messages ---


,message_id,patient_id,sender,message_text,sent_at



--- patients ---


,patient_id,first_name,last_name,date_of_birth,phone,email,created_at
0,1,Danielle,Johnson,1960-05-28,321.581.9600,danielle.johnson1@example.com,2024-05-08 12:51:38.269578
1,2,Carolyn,Hoffman,2000-02-04,001-486-537-9402x654,carolyn.hoffman2@example.com,2026-07-10 17:13:22.436696
2,3,Matthew,Davis,1944-05-10,(794)507-8161x849,matthew.davis3@example.com,2025-10-07 03:30:32.278760
3,4,Brandon,Perez,1961-09-12,731-564-7525,brandon.perez4@example.com,2025-08-30 11:47:45.019706
4,5,Jacqueline,Sutton,1942-01-29,+1-228-732-7648x3503,jacqueline.sutton5@example.com,2024-11-09 00:00:38.192081
5,6,Carl,Gentry,1986-09-12,001-653-876-7242x388,carl.gentry6@example.com,2026-05-06 16:50:52.240663
6,7,Reginald,Robinson,2025-06-07,671.201.2269x16697,reginald.robinson7@example.com,2025-09-10 13:22:20.461288
7,8,Danny,Dyer,2003-03-26,001-551-446-2704x82814,danny.dyer8@example.com,2024-10-10 04:02:21.042022
8,9,Evan,Ashley,2018-09-03,2955701543,evan.ashley9@example.com,2026-04-15 00:17:16.353272
9,10,Stacey,Miller,1979-05-23,882-527-8248,stacey.miller10@example.com,2026-06-09 17:42:00.470727



--- waitlist ---


,waitlist_id,patient_id,appointment_type,status,added_at
0,1,59,Vaccination,waiting,2026-08-23 12:21:17.092603
1,2,58,Physical Exam,waiting,2026-08-27 11:27:31.459252
2,3,7,Vaccination,waiting,2026-08-24 20:53:51.941339
3,4,22,Check-up,waiting,2026-08-27 01:30:31.851054
4,5,78,Follow-up,waiting,2026-08-27 07:52:42.039529
5,6,47,Follow-up,waiting,2026-08-22 21:06:07.787330
6,7,39,Check-up,waiting,2026-08-23 11:04:18.827104
7,8,88,Consultation,waiting,2026-08-21 05:37:29.559869
8,9,55,Check-up,waiting,2026-08-17 14:38:49.913030
9,10,28,Physical Exam,waiting,2026-08-17 17:51:40.753953


## Quick sanity checks

Confirms every canceled appointment has a valid patient, and every waitlist/message row points to a real patient.

In [10]:
patient_ids = set(dfs["patients"]["patient_id"])

orphan_calendar = dfs["calendar"][~dfs["calendar"]["patient_id"].isin(patient_ids)]
orphan_waitlist = dfs["waitlist"][~dfs["waitlist"]["patient_id"].isin(patient_ids)]
orphan_messages = dfs["messages"][~dfs["messages"]["patient_id"].isin(patient_ids)]

print(f"Orphan calendar rows: {len(orphan_calendar)}")
print(f"Orphan waitlist rows: {len(orphan_waitlist)}")
print(f"Orphan message rows: {len(orphan_messages)}")
print(
    f"Canceled appointments (pending agent work): {(dfs['calendar']['status'] == 'canceled').sum()}"
)

Orphan calendar rows: 0
Orphan waitlist rows: 0
Orphan message rows: 0
Canceled appointments (pending agent work): 6


In [11]:
conn.close()